In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

Prüfen der Daten im Volume

In [0]:
%python
dataset_school = "/Volumes/workspace/default/volume"

all_files = dbutils.fs.ls(dataset_school)
json_files = [f for f in all_files if f.name.endswith(".json")]

display(all_files)
display(json_files)

Querying JSON Data

In [0]:
%sql
SELECT * 
FROM json.`/Volumes/workspace/default/volume/students.json`; -- Repetition Variante 1

In [0]:
# Alternative (Variante 2):
students_df = spark.read.json("/Volumes/workspace/default/volume/students.json")
students_df.createOrReplaceTempView("students")

courses_df = spark.read.json("/Volumes/workspace/default/volume/courses.json")
courses_df.createOrReplaceTempView("courses")

enrollments_df = spark.read.json("/Volumes/workspace/default/volume/enrollments.json")
enrollments_df.createOrReplaceTempView("enrollments")

In [0]:
%sql
SELECT * FROM students;

In [0]:
%sql
SELECT * FROM courses;

In [0]:
%sql
SELECT * FROM enrollments;

Unstrukturierte Daten

In [0]:
image_path = "/Volumes/workspace/default/volume/*.png" 

df = spark.read.format("binaryFile").load(image_path)
display(df)

In [0]:
from PIL import Image
import io

# Laden mit binaryFile
image_path = "/Volumes/workspace/default/volume/image.png"
df_bin = spark.read.format("binaryFile").load(image_path)

# Inhalte extrahieren
binary_content = df_bin.select("content").first()[0]
image = Image.open(io.BytesIO(binary_content))
width, height = image.size

# DataFrame mit Metadaten erzeugen
df_meta = spark.createDataFrame(
    [(image_path, width, height)],
    ["origin", "width", "height"]
)
df_meta.createOrReplaceTempView("image_view")

In [0]:
# Zeige Bild im Notebook
image.show()

In [0]:
%sql
SELECT origin, width, height
FROM image_view

# csv ab Volume einlesen und als Delta-Tabelle abspeichern

In [0]:
# 1) Einlesen mit expliziter Schema-Option
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true") # Spark scannt die Datei beim Einlesen einmal durch und prüft: „Bestehen alle Einträge in dieser Spalte aus Zahlen?“ Wenn ja, ordnet Spark der Spalte automatisch den Datentyp Integer oder Double zu.
    .load("/Volumes/workspace/default/volume/sample_data.csv"))

# 2) Schreiben in den Unity Catalog (3-Level-Namespace)
(df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.my_csv_table"))

In [0]:
%sql
SELECT * FROM workspace.default.my_csv_table

In [0]:
%sql
DESCRIBE EXTENDED workspace.default.my_csv_table

# Insert, Update, Delete

In [0]:
%sql
INSERT INTO default.my_csv_table VALUES (4, 'Dora', 28, 'Basel');


In [0]:
%sql
UPDATE default.my_csv_table
SET city = 'Zurich'
WHERE id = 1;   -- Alice

In [0]:
%sql
DELETE FROM default.my_csv_table
WHERE id = 2;   -- Bob

# Time Travel

In [0]:
%sql
DESCRIBE HISTORY default.my_csv_table;

In [0]:
%sql
SELECT * 
FROM default.my_csv_table VERSION AS OF 0;

In [0]:
%sql
SELECT * 
FROM default.my_csv_table VERSION AS OF 2;

Restore Version 0

In [0]:
%sql
-- Tabelle auf Version 0 zurücksetzen
RESTORE TABLE default.my_csv_table TO VERSION AS OF 0;

-- Kontrolle: aktueller Zustand nach Restore
SELECT * FROM default.my_csv_table;

# Datei-Update

Achtung keine ACID Garantien!

In [0]:
# 1) Einlesen der Datei mit Schema-Erkennung
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true") 
    .load("/Volumes/workspace/default/volume/sample_data.csv"))

# 2) Struktur der Daten prüfen
print("Schema der Daten (Datentypen):")
df.printSchema()

# 3) Die ersten Zeilen anzeigen
print("Inhalt der Daten:")
df.show(5)

In [0]:
from pyspark.sql.functions import current_date

# Füge dem DataFrame eine neue Spalte hinzu
df_neu = df.withColumn("hinzugefuegt_am", current_date())

# Zeige das geänderte DataFrame an
print("Geänderte Daten:")
df_neu.show()

In [0]:
# Ziel ist ein VERZEICHNIS, keine einzelne Datei
target_path = "/Volumes/workspace/default/volume/sample_data_neu"

(df_neu.write
    .format("csv")
    .mode("overwrite")
    .option("header", "true")
    .option("sep", ";")
    .save(target_path))

print(f"Daten wurden erfolgreich im Verzeichnis '{target_path}' gespeichert.")

In [0]:
from pyspark.sql.functions import when, col

# 1) Laden mit korrekten Typen (wichtig für den Vergleich!)
# Wir lesen aus dem Pfad, den wir vorher als Ordner angelegt haben
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/volume/sample_data_neu"))

# 2) Transformation mit 'when' und 'col'
# Wir nutzen die funktionale API von Spark
df_updated = df.withColumn(
    "city", 
    when(col("name") == "Bob", "Updated City").otherwise(col("city"))
)

# 3) Interaktive Anzeige statt Text-Output
print("Daten nach dem Update:")
display(df_updated)

In [0]:
# Zielpfad definieren (als Verzeichnis/Ordner)
target_path = "/Volumes/workspace/default/volume/sample_data_neu"

# 1) Das Speichern im Spark-Format (Erzeugt einen Ordner mit CSV-Teilen)
(df_updated.write
    .format("csv")
    .mode("overwrite")
    .option("header", "true")
    .option("sep", ";")
    .save(target_path))

print(f"Das Verzeichnis '{target_path}' wurde mit den aktualisierten Daten überschrieben.")

In [0]:
# 1) Einlesen der aktualisierten Daten
df_aktuell = (spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/volume/sample_data_neu"))

# 2) Validierung der Änderungen
print("Finale Kontrolle der Daten im Volume:")
display(df_aktuell)

In [0]:
import os

dataset_school = "/Volumes/workspace/default/volume/sample_data_neu"
# Listet alle Einträge auf und filtert nach .csv
files = [f for f in os.listdir(dataset_school) if f.endswith(".csv")]

print(f"Gefundene CSV-Objekte: {files}")

# Aufräumen

In [0]:
%sql
DROP TABLE my_csv_table;

In [0]:
# Den gesamten Ordner "sample_data_neu" im Volume unwiderruflich löschen
# Der Parameter 'True' sorgt dafür, dass auch alle Dateien im Ordner gelöscht werden
dbutils.fs.rm("/Volumes/workspace/default/volume/sample_data_neu", True)

print("Der gesamte Ordner wurde erfolgreich aus dem Volume entfernt.")

In [0]:
%python
# Prüfen der csv im volume
dataset_school = "/Volumes/workspace/default/volume"

all_files = dbutils.fs.ls(dataset_school)
csv_files1 = [f for f in all_files if f.name.endswith(".csv")]
csv_files2 = [f for f in all_files if f.name.endswith(".csv/")]

display(csv_files1)
display(csv_files2)